In [1]:
import sys
sys.path.append('..')
import os
import numpy as np
import pandas as pd
import csv
import io
import time
import re
import math
import import_ipynb
from Bio.KEGG import REST
from tqdm import tqdm
from collections import defaultdict
from intermine.webservice import Service
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
from bokeh.io import show, output_notebook, output_file
from bokeh.models import ColumnDataSource, LinearColorMapper, ColorBar, Legend, BasicTicker, LabelSet
from bokeh.plotting import figure
from bokeh.transform import factor_cmap, transform
from bokeh.palettes import Turbo256, viridis
csv.field_size_limit(1000000)
output_notebook(hide_banner=True)
       

intro = "Welcome to the pathway enrichment data formatting.<div> Please indicate if you want to submit a query to Mouse Mine by loading a list of genes or if you are loading the data from your PC."
sep = "Or load a file:"
desc = "After loading the data, choose one of the different plots:<ol><li>Barplot where for each KEGG pathway ID we have associated a subtable containing the MGI and ENSEMBL gene ID found in the pathway and the phenotypes associated to each gene.</li><li>Heatmap where for each MGI gene ID we have the associated list of pathways and phenotypes.</li><li>Spiderchart where for each MP phenotype id, the list of genes related to them and the pathways in which they were observed by the KOBAS algorithm.</li></ol><div>Select one of the following methods:"
desc2 = "Please load a list of path IDs or GO terms to be showed:"
desc3 = "Please load a list of phenotypes (with ID in the format MP:0000000):"
clear_output(wait=False)

In [ ]:
class App:
    
    def __init__(self):
        self._plot_container = widgets.Output()
        
        #Loading section
        self.load_q_button, load_q_box = self._create_query_button()
        self.text_a, text_box = self._create_text_area()
        _load_container = widgets.VBox([widgets.HTML(intro,layout = widgets.Layout(margin='0 0 0 2em')),
                                        text_box,
                                        widgets.HTML(sep,layout = widgets.Layout(margin='0 0 0 2em')),
                                        load_q_box], layout = widgets.Layout(align_items='center', flex ='3 0 auto'))
        
        #Data analysis selection section
        self.path_but, self.path_box = self._create_path()
        self.gene_but, self.gene_box = self._create_gene()
        self.spid_but, self.spid_box = self._create_spid()
        self._buttons_container = widgets.HBox([], layout = widgets.Layout(align_items='center', flex ='3 0 auto'))
        self._analysis_container = widgets.VBox([widgets.HTML(desc,layout = widgets.Layout(margin='0 0 0 2em')),self._buttons_container], layout = widgets.Layout(align_items='center', flex ='3 0 auto'))
    
        
        #Final UI composition section
        self.container = widgets.VBox([
            widgets.HTML(('<h1><em><b>Gene pathway enrichment</b></em></h1>'),layout=widgets.Layout(align_items='center')),
            _load_container,
            self._analysis_container,
            self._plot_container], layout = widgets.Layout(flex='3 0 auto', margin='0 auto 0 auto', max_width='2048px'))
    
    
    #Load section functions
    
    def _create_text_area(self):
        text_a = widgets.Text(placeholder='Ex: MGI:00001,Kras,...', description='Genes list:')
        text_a.on_submit(self._on_change_a)
        text_box = widgets.HBox([text_a])
        return (text_a,text_box)
    
    def _create_query_button(self):
        load_b = widgets.FileUpload(accept=".txt, .csv, .dat", multiple=False, description = "Load query list")
        load_b.observe(self._on_change_q, names=['value'])
        load_box = widgets.HBox([load_b])
        return (load_b, load_box)
    
    #Functions to work loaded data
    def _on_change_a(self,_):
        self.text_a.disabled = True
        inp = list(re.split(',|\n|\s|\t',self.text_a.value))
        self._on_loaded_input(inp)
        
    def _on_change_q(self,_):
        inp = list(self.load_q_button.value.values())[0] #if multiple setted to true in FileUpload(), will not work!
        content = inp['content']
        content = io.StringIO(content.decode('utf-8'))
        genes = pd.read_csv(content,header=None).values
        genes = np.array2string(genes)
        self._on_loaded_input(genes)
            
    def _on_loaded_input(self,genes):    
        with self._plot_container:
            
            self._plot_container.clear_output(wait=True)
            print("Querying MouseMine, please wait", end="")
            service = Service("https://www.mousemine.org/mousemine/service")
            print(".",end="")
            query = service.new_query("Allele")
            query.add_constraint("ontologyAnnotations.ontologyTerm", "MPTerm")
            query.add_view("primaryIdentifier", "symbol", "name", "ontologyAnnotations.ontologyTerm.identifier",
                           "ontologyAnnotations.ontologyTerm.name", "feature.symbol", "feature.primaryIdentifier")
            query.add_constraint("organism.taxonId", "=", "10090", code="B")
            query.add_constraint("Allele", "LOOKUP", ','.join(genes), code="D")

            self.mouseMine = pd.DataFrame(np.NaN, index=range(0, len(query)),
                                     columns=["Allele > Primary Identifier", "Allele > Symbol", "Allele > Name",
                                              "Allele > Ontology Annotations > Ontology Term . Identifier",
                                              "Allele > Ontology Annotations > Term Name", "Allele > Gene",
                                              "Allele > Feature > Primary Identifier"])
            print(".",end="")
            for row, i in zip(query.rows(), range(0, len(query))):
                self.mouseMine.loc[[i]] = [row["primaryIdentifier"], row["symbol"], row["name"],
                                      row["ontologyAnnotations.ontologyTerm.identifier"],
                                      row["ontologyAnnotations.ontologyTerm.name"], row["feature.symbol"],
                                      row["feature.primaryIdentifier"]]
            print(".")
            print("Query ended successfully")
            
            #Load KOBAS data mapping
            self.KOBAS_identify = pd.read_csv("../modules/KOBAS_identify.csv", header=0, sep="\t", index_col=[0])
            self.KOBAS_identify = self.KOBAS_identify.reset_index(drop=True)
            self.p = pd.DataFrame()
            self.g = pd.DataFrame()
            self._buttons_container.children = tuple(list(self._buttons_container.children)+[self.path_box,self.gene_box,self.spid_box])
            self._update_app()
    
    #Data analysis section functions    
    def _create_path(self):    
        button = widgets.Button(description="Barplot")
        button.on_click(self._on_click_path)
        box = widgets.HBox([button])
        return (button, box)
        
    def _create_gene(self):
        button = widgets.Button(description="Heatmap")
        button.on_click(self._on_click_gene)
        box = widgets.HBox([button])
        return (button, box)
    
    def _create_spid(self):
        button = widgets.Button(description="Spiderchart")
        button.on_click(self._on_click_spid)
        box = widgets.HBox([button])
        return (button, box)
        
    def _on_click_path(self, _):
        self._plot_container.clear_output(wait=True)
        with self._plot_container:
            self.p, self.p_list = path_dataset(self.mouseMine, self.KOBAS_identify)
        self._analysis_container.children = tuple(list(self._analysis_container.children)+[widgets.HTML(desc2,layout = widgets.Layout(margin='0 0 0 2em'))])
        self.load_t_button, term_box = self._create_term()
        self._analysis_container.children = tuple(list(self._analysis_container.children)+[term_box])
        clear_output(wait=True)
        
                                                                                                   
    def _create_term(self):                                                                        
        self._plot_container.clear_output(wait=False)
        load_b = widgets.FileUpload(accept=".txt, .csv, .dat", multiple=False, description = "Load Terms")
        load_b.observe(self._on_change_terms, names=['value'])
        load_box = widgets.HBox([load_b])
        return (load_b, load_box)
    
    def _create_phen(self):
        load_b = widgets.FileUpload(accept=".txt, .csv, .dat", multiple=False, description = "Load Phenotypes")
        load_b.observe(self._on_change_phen, names=['value'])
        load_box = widgets.HBox([load_b])
        return (load_b, load_box)

    def _on_change_terms(self,_):
        try:
            with self._plot_container:
                self._plot_container.clear_output(wait=True)
                inp = list(self.load_t_button.value.values())[0]
                content = inp['content']
                content = io.StringIO(content.decode('utf-8'))
                terms = pd.read_csv(content,header=None).values
                terms = terms.tolist()
                self.p_plot = bar_plot_phen(self.p,self.p_list,terms)
                self._plot_container.clear_output(wait=True)
                show(self.p_plot, notebook_handle = True)
        except (NameError,AttributeError) as e:
            print(e)
            pass
        
    def _on_change_phen(self,_):
        try:
            with self._plot_container:
                self._plot_container.clear_output(wait=True)
                inp = list(self.load_g_button.value.values())[0]
                content = inp['content']
                content = io.StringIO(content.decode('utf-8'))
                phen = pd.read_csv(content,header=None).values
                phen = phen.tolist()
                self.g_plot = spiderchart_paths(self.g,self.g_list,phen)
                self._plot_container.clear_output(wait=True)
                show(self.g_plot, notebook_handle = True)
        except (NameError,AttributeError) as e:
            print(e)
            pass
    

    def _on_click_gene(self, _):
        try:
            with self._plot_container:
                self._plot_container.clear_output(wait=True)
                self.g = gene_dataset(self.mouseMine, self.KOBAS_identify)
                self.g_plot = genes_phen_plot(self.g)
                self._plot_container.clear_output(wait=True)
                show(self.g_plot, notebook_handle = True)
        except (NameError,AttributeError) as e:
            pass    
        
    def _on_click_spid(self,_):    
        try:
            with self._plot_container:
                self._plot_container.clear_output(wait=True)
                self.g, self.g_list = path_dataset(self.mouseMine, self.KOBAS_identify)
                self._analysis_container.children = tuple(list(self._analysis_container.children)+[widgets.HTML(desc3,layout = widgets.Layout(margin='0 0 0 2em'))])
                self.load_g_button, phen_box = self._create_phen()
                self._analysis_container.children = tuple(list(self._analysis_container.children)+[phen_box])
        except (NameError,AttributeError) as e:
            pass
        
    #Shared functions
    def _on_change(self, _):
        self._update_app()
        
    def _update_app(self):
        self._plot_container.clear_output(wait=True)
        
    
    def gene_dataset(mouseMine,KOBAS_identify):
        pd.options.mode.chained_assignment = None

        #Create the list of genes for the final table
        genes = pd.DataFrame(columns=['MGI_Gene_ID','Path','Phenotype'])
        genes.MGI_Gene_ID = mouseMine['Allele > Feature > Primary Identifier']
        genes = genes.drop_duplicates(ignore_index=True)

        #Creating a dictionary with genes as key and path id as values, used for faster comuputations
        df = KOBAS_identify[['ID','Input_MGI']]
        for i in range(0, len(df)):
            df.loc[i]['Input_MGI'] = (df.loc[i]['Input_MGI'].translate({ord(i): None for i in '[]\'\n'})).split(',')

        d = defaultdict(list)
        for idx, row in df.iterrows():
            gen = row['Input_MGI']
            for g in gen:
                d[g].append(row['ID'])

        #Creating the list of pathway for each gene (using the whole mousemine database takes up to 3/4 min)
        for i in tqdm(range(0,len(genes))):
            if (genes['MGI_Gene_ID'][i] in d.keys()):
                genes.Path[i] = d[genes['MGI_Gene_ID'][i]]
            genes.Phenotype[i] = mouseMine['Allele > Ontology Annotations > Ontology Term . Identifier'][mouseMine['Allele > Feature > Primary Identifier'] == genes.MGI_Gene_ID[i]].values

        genes = genes[genes['Phenotype'].notna()]
        genes.reset_index(inplace=True,drop=True)

        return(genes)
    
    
    def path_dataset(mouseMine, KOBAS_identify):
        pd.options.mode.chained_assignment = None

        # Creating the lists of genes id
        ens_lists = KOBAS_identify['Input']
        mgi_lists = KOBAS_identify['Input_MGI']

        # Takes from mouseMine only the columns that I need and create a dataframe with all the pathways gruped per gene id
        mM = mouseMine[
            ['Allele > Ontology Annotations > Ontology Term . Identifier', 'Allele > Feature > Primary Identifier']]
        mM2 = mM.groupby(['Allele > Feature > Primary Identifier'], sort=False)

        # to speed up the search of the phenotypes related to each gene inside the lists, it's created a dictionary containing
        # a gene as key and list of phen_id related to it
        d = {}
        genes = mouseMine.drop_duplicates('Allele > Feature > Primary Identifier', inplace=False, ignore_index=True)
        genes = genes['Allele > Feature > Primary Identifier']

        for i in range(0, len(genes)):
            d[genes[i]] = mM['Allele > Ontology Annotations > Ontology Term . Identifier'][mM2.groups[genes[i]]].values


        # Create the subdataframe of each pathway, most time consuming step takes up to 1/2 min to complete
        df_list = []
        df = pd.DataFrame(columns=['MGI_ID', 'ENS_ID', 'Pheno_list'])
        for i in tqdm(range(0, len(ens_lists))):
            df.MGI_ID = (mgi_lists[i].translate({ord(i): None for i in '[]\'\n'})).split(',')
            df.MGI_ID = df.MGI_ID.str.strip()
            df.ENS_ID = (ens_lists[i].translate({ord(i): None for i in '[]\'\n'})).split(',')
            df.ENS_ID = df.ENS_ID.str.strip()
            for j in df.MGI_ID:
                if j in d.keys():
                    df['Pheno_list'][df['MGI_ID'][df['MGI_ID'] == j].index[0]] = d[j]
            df_list.append(df)
            df = df.iloc[0:0]
        del (df, i)

        # Remove lines with Nan values (genes not found in the list)
        for i in range(0,len(df_list)):
            df_list[i] = df_list[i].dropna()
            df_list[i] = df_list[i].reset_index(drop=True)

        # Combining all the info into the final table

        path_data = pd.DataFrame(columns=['Pathway', 'Path_ID', 'Pheno_data'])
        path_data['Pathway'] = KOBAS_identify.Term
        path_data['Path_ID'] = KOBAS_identify.ID
        for i in range(0, len(df_list)):
            path_data['Pheno_data'][i] = pd.DataFrame(df_list[i])

        d = []

        #Removing the rows with only empty subdataframes and converting the Pheno_list fields from series to array
        for i in range(0, len(path_data)):
            if path_data.loc[i]['Pheno_data'].empty:
                d.append(i)
        path_data.drop(d, inplace=True)
        path_data.reset_index(drop=True, inplace=True)
        del (d, i)

        return path_data, df_list
    
    
    #PLOT FUNCTION SECTION (NB: TO BE MOVED IN SEPARATED MODULES)


    def genes_phen_plot(g):
        

        # Taking the lists of genes and phenotypes
        gn = [i for i in g['MGI_Gene_ID']]
        ph = [j for i in g['Phenotype'] for j in i]

        # remove the duplicates from both the lists
        seen = set()
        seen_add = seen.add
        genes = [x for x in gn if not (x in seen or seen_add(x))]

        seen = set()
        seen_add = seen.add
        phen = [x for x in ph if not (x in seen or seen_add(x))]
        del (ph, gn, seen, seen_add)

        # Dictionary where the keys are genes and values the phenotypes related to them
        d = {}
        for i in range(0, len(genes)):
            d[genes[i]] = g['Phenotype'][g['MGI_Gene_ID'] == genes[i]].values[0]

        # Generating matrix where we have '1' if the phenotype is expressed by the gene (y axis phen, x axis genes)
        presence = np.zeros((len(phen), len(genes)))
        for i in range(len(phen)):
            for j in range(len(genes)):
                if phen[i] in d[genes[j]]:
                    presence[i, j] = 1
        del (i, j)

        # formatting the data to use it with bokeh heatmap
        df = pd.DataFrame(presence, columns=genes, index=phen)
        df.index.name = 'Phenotypes'
        df.columns.name = 'Genes'
        df = df.stack().rename("value").reset_index()

        # colors used for the heatmap, you can use your own palette here (only 2 colors needed)
        colors = ['#1926d7', '#d71919']

        # Specific mapper to map color with value
        mapper = LinearColorMapper(palette=colors, low=df.value.min(), high=df.value.max())

        # Define a figure
        fig = figure(
                    plot_width=1920,
                    plot_height=1080,
                    title="Genes to phenotypes relation",
                    x_range=list(df.Genes.drop_duplicates()),
                    y_range=list(df.Phenotypes.drop_duplicates()),
                    toolbar_location='above',
                    tools='pan,box_zoom,reset,save',
                    x_axis_location="above",
                    tooltips=[('Gene', '@Genes'), ('Phenotype', '@Phenotypes'), ('Relation', '@value')])

        fig.yaxis.major_label_orientation= math.pi/4
        fig.xaxis.major_label_orientation = math.pi / 4

        # Create rectangle for heatmap
        fig.rect(
                    x="Genes",
                    y="Phenotypes",
                    width=1,
                    height=1,
                    source=ColumnDataSource(df),
                    line_color="black",
                    fill_color=transform('value', mapper))

        # Add legend
        color_bar = ColorBar(
                    color_mapper=mapper,
                    location=(0, 0),
                    ticker=BasicTicker(desired_num_ticks=len(colors)),
                    major_label_overrides={'1': '1 - Correlated', '0.5': '', '0': '0 - Not correlated'}
                )

        fig.add_layout(color_bar, 'right')
        return fig

    def spiderchart_paths(p,p_list,phen):

        #Load the ENtrez id - MGI id mapping table
        ent_mgi = pd.read_table("../modules/ENT_MGI.csv", header=0,index_col=[0])
        ent_mgi = ent_mgi.reset_index(drop=True)
        ent_mgi['ENTREZ_ID'] = ent_mgi['ENTREZ_ID'].astype('object')

        #Load KEGG pathway modules table and convert it in a dictonary
        KEGG_modules_mapping = pd.read_table("../modules/KEGG_modules.csv",header=0, index_col=[0], sep="\t")
        mod_cat={}
        for i in range(0,len(KEGG_modules_mapping)):
            mod_cat[KEGG_modules_mapping['Mod_ID'][i]] = KEGG_modules_mapping['Macro_ID'][i]

        #Load the main category mapping table
        mod_cat_ID = pd.read_table("../modules/main_Modules.csv", header=0, index_col=[0], sep="\t")

        # Use the Biopython KEGG module to get the KEGG pathway module IDs related to mouse genes
        x = REST.kegg_link('module','mmu')
        y = x.read()
        y = io.StringIO(y)
        mod_genes = pd.read_csv(y, sep='\t', header=None, names=('ENTREZ_ID', 'Mod_ID'))
        del(x,y)

        #Convert entrez id to mgi id in mod_genes
        #First we need to remove the string "mmu:" in the KEGG_ID column
        mod_genes['ENTREZ_ID'] = mod_genes['ENTREZ_ID'].map(lambda x: x.lstrip('mmu:'))

        #mapping mod_genes (ISSUE: for some reason we need to convert the ENTREZ_ID columns into string type since the object type is not working for the merge)
        mod_genes['ENTREZ_ID'] = mod_genes['ENTREZ_ID'].astype(str)
        ent_mgi['ENTREZ_ID'] = ent_mgi['ENTREZ_ID'].astype(str)
        mapped = pd.merge(mod_genes,ent_mgi, on=['ENTREZ_ID'])
        mapped = mapped.drop(['ENTREZ_ID'], axis=1)

        # Group by genes ID (and convert it into a dictonary to speed up search), in this way we'll obtain a list of modules for each gene
        mapped = mapped.groupby('MGI_ID')['Mod_ID'].apply(list).to_frame().reset_index()
        m = mapped.set_index('MGI_ID').T.to_dict('list')

        for i, row in p.iterrows():
            p['Pheno_data'][i]['Mod_ID']=np.NaN
            for j in row['Pheno_data']['MGI_ID']:
                if j in m.keys():
                    row['Pheno_data']['Mod_ID'][row['Pheno_data']['MGI_ID'][row['Pheno_data']['MGI_ID']==j].index[0]] = m[j][0]
                    print("True for " + j + " in position " + str(i))

        df = pd.DataFrame(columns=['Pheno_ID','Genes_count','Norm_count'])
        df['Pheno_ID'] = phen
        for i in range(0,len(df)):
            df['Genes_count'][i] = [0,0,0,0,0,0,0,0,0,0]
            df['Norm_count'][i] = [0,0,0,0,0,0,0,0,0,0]


        big_list = pd.concat(p_list).dropna().reset_index(drop=True).drop_duplicates(subset="MGI_ID", keep="first",inplace=False,ignore_index=True)
        for i in range(0,len(df)):
            for j in range(0,len(big_list)):
                if df['Pheno_ID'][i] in big_list['Pheno_list'][j]:
                    for l in big_list['Mod_ID'][j]:
                        try:
                            if mod_cat[l] != 'X' or mod_cat[l] != '?':
                                df['Genes_count'][i][int(mod_cat[l])-1] += 1
                        except KeyError:
                            continue

        max = np.max(list(df['Genes_count']))
        min = np.min(list(df['Genes_count']))
        for i in range(0,len(df)):
            for j in range(0,len(df['Norm_count'][i])):
                df['Norm_count'][i][j] = (df['Genes_count'][i][j] - min) / (max-min)

        #GENERATING THE SPIDERCHART

        n_vertex = len(mod_cat_ID)-2
        theta = np.linspace(0, 2 * np.pi, n_vertex, endpoint=False)

        # rotate theta such that the first axis is at the top
        theta += np.pi / 2

        centre = 1

        #Labels for the circles to indicate the counting
        lab = pd.DataFrame(columns=['x', 'y', 'names'])
        lab['x'] = [0.2, 0.4, 0.6, 0.8, 1]
        lab['y'] = [centre,centre,centre,centre, centre]
        interval = np.linspace(0, max, 6)
        for i in range(0,len(interval)):
            interval[i] = int(interval[i])
        lab['names'] = interval[1:6]
        del (interval)

        def unit_poly_verts(theta, centre):
            """Return vertices of polygon for subplot axes.
            This polygon is circumscribed by a unit circle centered at (centre, centre)
            """
            x0, y0, r = [centre] * 3
            verts = [(r * np.cos(t) + x0, r * np.sin(t) + y0) for t in theta]
            return verts

        def radar_patch(r, theta, centre):
            """ Returns the x and y coordinates corresponding to the magnitudes of
            each variable displayed in the radar plot
            """
            # offset from centre of circle
            offset = 0
            yt = (r * centre + offset) * np.sin(theta) + centre
            xt = (r * centre + offset) * np.cos(theta) + centre
            return xt, yt

        verts = unit_poly_verts(theta, centre)
        x = [v[0] for v in verts]
        y = [v[1] for v in verts]
        x.append(verts[0][0])
        y.append(verts[0][1])

        fig = figure(title="Phenotype/Pathway module - Radar plot",height = 1980,width = 1080,toolbar_location='above',tools='pan,box_zoom,reset,save')
        text = mod_cat_ID['Name'][0:10].append(pd.Series(mod_cat_ID['Name'][0]))
        source = ColumnDataSource({'x': x , 'y': y , 'text': text})

        fig.line(x="x", y="y", source=source)

        labels = LabelSet(x="x", y="y", text="text", source=source)
        fig.add_layout(labels)

        cols = viridis(len(df))

        for i in range(len(df)):
            xt, yt = radar_patch(np.asarray(df['Norm_count'][i]), theta, centre)
            fig.patch(x=xt, y=yt, fill_alpha=0.15, fill_color=cols[i], line_color=cols[i], legend_label=str(df['Pheno_ID'][i]))

        # circular axes and lables
        radii = [0,0.2,0.4,0.6,0.8]
        fig.circle(centre, centre, radius=radii, fill_color=None, line_color="gray", line_alpha=0.15)
        fig.text(centre, radii, [str(r) for r in reversed(lab['names'])], text_font_size="5pt", text_align="center", text_baseline="middle", text_alpha=0.2)

        fig.axis.visible = False
        fig.xgrid.grid_line_color = None
        fig.ygrid.grid_line_color = None
        legend = Legend(items=fig.legend.items)
        fig.legend.visible=False
        legend.click_policy = 'hide'
        legend.visible=True
        fig.add_layout(legend,"left")
        return fig

    def bar_plot_phen(p,p_list,terms):

        #add a column with the count of phenotype and one with the count of genes for each pathway
        p['phen_count'] = pd.NA
        p['gene_count'] = pd.NA
        for i in range(0,len(p)):
            if len(p_list[i]['Pheno_list']) == 0:
                p['phen_count'][i] = 0
            else:
                p['phen_count'][i] = len(list(dict.fromkeys(p_list[i]['Pheno_list'].apply(pd.Series).stack().reset_index(drop=True).to_list())))
            p['gene_count'][i] = len(p_list[i]['MGI_ID'])

        d = {k: v for v, k in enumerate(p.iloc[:,1].to_list())}
        l2 = []
        tmp = p.iloc[:,1]
        for i in terms:
            if i in tmp.unique():
                l2.append(p.iloc[d[i[0]]])
        df = pd.DataFrame(l2).reset_index(drop=True)
        #factors used by bokeh for the count colors for genes
        fac = []
        for i in range(0,len(df)):
            fac.append(str(df['gene_count'][i]))
        fac = pd.Series(fac)
        #here  if the number of count it's bigger than the size of the color palette, we are going to
        #divide the counts and colors into ranges. Otherwise we are going to use the default palette
        if len(fac.unique()) > 256:
            step = df['gene_count'].max() / 256
            bounds = [round(step * i) for i in range(256)]
            bound_colors = []
            j = 0
            x = False
            for i in range(0, df['gene_count'].max()):
                if not x:
                    if i >= bounds[j + 1]:
                        j += 1
                        if j+1 == 255:
                            x=True
                    bound_colors.append(Turbo256[j])
                else:
                    bound_colors.append(Turbo256[255])
            source = ColumnDataSource(data=dict(path=df.iloc[:,1], counts=df['phen_count'], g_counts=fac))
            my_cmap = LinearColorMapper(palette=bound_colors, low=0, high=df['gene_count'].max())


            fig = figure(title="Phenotype/genes count per pathway",width=1280, height=720, x_range=df.iloc[:,1], tooltips=[('Gene count', '@g_counts'), ('Phenotype count', '@counts')],toolbar_location='above',tools='pan,box_zoom,reset,save')
            fig.vbar(x='path', top='counts', width=0.9, source=source, line_color='white', fill_color={"field":"g_counts","transform":my_cmap})
            fig.xgrid.grid_line_color = None
            fig.y_range.start = 0
            fig.y_range.end = np.max(df['phen_count'].values) + 100
            fig.xaxis.major_label_orientation = math.pi / 2
        else:
            source = ColumnDataSource(data=dict(path=df.iloc[:,1], counts=df['phen_count'], g_counts=fac))
            colors = factor_cmap('g_counts', palette=viridis(len(fac.unique())), factors=fac.unique())
            fig = figure(title="Phenotype/genes count per pathway", width=1280, height=720, x_range=df.iloc[:,1], tooltips=[('Gene count', '@g_counts'), ('Phenotype count', '@counts')],toolbar_location='above',tools='pan,box_zoom,reset,save')
            fig.vbar(x='path', top='counts', width=0.9, source=source, line_color='white', fill_color=colors)
            fig.xgrid.grid_line_color = None
            fig.y_range.start = 0
            fig.y_range.end = np.max(df['phen_count'].values) + 100
            fig.xaxis.major_label_orientation = math.pi / 2


        return fig


In [ ]:
app = App()
app.container

In [4]:
print(sys.path)

['/Users/andrea/Desktop/Jupyter_interactive_plots/Jupyther_interactive_plots/MGI_interactive', '/opt/miniconda3/lib/python37.zip', '/opt/miniconda3/lib/python3.7', '/opt/miniconda3/lib/python3.7/lib-dynload', '', '/opt/miniconda3/lib/python3.7/site-packages', '/opt/miniconda3/lib/python3.7/site-packages/IPython/extensions', '/Users/andrea/.ipython', '..', 'Jupyter_interactive_plots/modules/']


In [3]:
sys.path.append('Jupyter_interactive_plots/modules/')

In [8]:
self.KOBAS_identify = pd.read_csv("/Users/andrea/Desktop/Jupyter_interactive_plots/Jupyther_interactive_plots/modules/KOBAS_identify.csv", header=0, sep="\t", index_col=[0])

NameError: name 'self' is not defined